# Superstore Sales — Exploratory Data Analysis

This notebook explores retail transaction data from a US-based superstore covering 4 years (2014–2017). The goal is to understand what drives sales and profit, where the business is doing well, and where it's losing money.

**Sections:**
1. The Data
2. Sales and Profit Overview
3. Categories and Products
4. Regional and State Performance
5. Customer Segments
6. Discount Analysis
7. Time Trends and Seasonality
8. Shipping Analysis
9. Customer Loyalty
10. Summary and Recommendations


## 1. The Data

Load the dataset, clean column names, parse dates, and create a few useful columns before the analysis begins.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

df = pd.read_csv("superstore.csv", encoding="latin1")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"]  = pd.to_datetime(df["ship_date"])

df["year"]          = df["order_date"].dt.year
df["quarter"]       = df["order_date"].dt.quarter
df["month"]         = df["order_date"].dt.month
df["shipping_days"] = (df["ship_date"] - df["order_date"]).dt.days
df["profit_margin"] = df["profit"] / df["sales"]

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
print(f"Unique customers: {df['customer_id'].nunique()}")
print(f"Unique products:  {df['product_id'].nunique()}")


In [ ]:
df[["order_date","category","sub-category","region","segment","sales","quantity","discount","profit"]].head(8)


In [ ]:
# Check for missing values and duplicates
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nDuplicate rows: {df.duplicated().sum()}")


In [ ]:
df[["sales", "quantity", "discount", "profit"]].describe().round(2)


**What we have:**
- About 10,000 transactions across 2014 to 2017
- 3 product categories, 17 sub-categories, 4 US regions
- 793 unique customers, most of whom are repeat buyers
- No missing data


## 2. Sales and Profit Overview

Before breaking things down, it helps to understand the overall shape of the two most important numbers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Distribution of Sales and Profit")

sns.histplot(df["sales"], bins=40, kde=True, ax=axes[0])
axes[0].set_title("Sales per Order")
axes[0].set_xlabel("Sales ($)")
axes[0].axvline(df["sales"].median(), color="red", linestyle="--",
                label=f'Median: ${df["sales"].median():.0f}')
axes[0].legend()

sns.histplot(df["profit"], bins=40, kde=True, ax=axes[1])
axes[1].set_title("Profit per Order")
axes[1].set_xlabel("Profit ($)")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5, label="Break-even")
axes[1].axvline(df["profit"].median(), color="orange", linestyle="--",
                label=f'Median: ${df["profit"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()


Most orders are under $500 — this is a high-volume, relatively low-ticket business. The profit chart has a left tail, meaning a portion of orders are loss-making. The red line at zero shows roughly 19% of all orders do not make a profit.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Outlier Check")

sns.boxplot(y=df["sales"], ax=axes[0])
axes[0].set_title("Sales — a few very large orders")
axes[0].set_ylabel("Sales ($)")

sns.boxplot(y=df["profit"], ax=axes[1])
axes[1].set_title("Profit — some large losses too")
axes[1].set_ylabel("Profit ($)")

plt.tight_layout()
plt.show()

print(f"Orders with negative profit: {(df['profit'] < 0).sum():,} ({(df['profit']<0).mean()*100:.1f}%)")
print(f"Largest single loss:  ${df['profit'].min():,.0f}")
print(f"Largest single gain:  ${df['profit'].max():,.0f}")


## 3. Categories and Products

Not all categories are equally profitable. High sales do not always mean high profit.

In [ ]:
cat_df = df.groupby("category")[["sales","profit"]].sum().reset_index()
cat_df["margin"] = (cat_df["profit"] / cat_df["sales"] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Category Performance")

x = range(len(cat_df))
w = 0.35
axes[0].bar([i - w/2 for i in x], cat_df["sales"]/1000, w, label="Sales ($K)", alpha=0.85)
axes[0].bar([i + w/2 for i in x], cat_df["profit"]/1000, w, label="Profit ($K)", alpha=0.85)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(cat_df["category"])
axes[0].set_ylabel("$ Thousands")
axes[0].set_title("Sales vs Profit by Category")
axes[0].legend()

bars = axes[1].bar(cat_df["category"], cat_df["margin"], width=0.5)
axes[1].set_ylabel("Profit Margin (%)")
axes[1].set_title("Profit Margin % by Category")
axes[1].axhline(cat_df["margin"].mean(), color="grey", linestyle="--",
                label=f'Avg: {cat_df["margin"].mean():.1f}%')
axes[1].legend()
for bar, val in zip(bars, cat_df["margin"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val}%", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()


Furniture generates similar revenue to Technology but keeps only 2.5 cents per dollar compared to Technology's 17 cents. This is largely a discounting issue, which Section 6 explains. Office Supplies and Technology are the real profit drivers.


In [ ]:
sub = df.groupby("sub-category")[["sales","profit"]].sum().reset_index()
sub["margin"] = sub["profit"] / sub["sales"] * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Sub-Category Analysis")

sub_sorted = sub.sort_values("margin")
axes[0].barh(sub_sorted["sub-category"], sub_sorted["margin"])
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("Profit Margin (%)")
axes[0].set_title("Profit Margin by Sub-Category (ranked)")

sub_top = sub.sort_values("sales", ascending=False).head(10)
axes[1].barh(sub_top["sub-category"], sub_top["sales"]/1000)
axes[1].set_xlabel("Total Sales ($K)")
axes[1].set_title("Top 10 Sub-Categories by Sales")

plt.tight_layout()
plt.show()

print("Loss-making sub-categories:")
print(sub[sub["margin"] < 0][["sub-category","sales","profit","margin"]].sort_values("margin").to_string(index=False))


Tables appear strong with $207K in sales but are actually losing $17.7K — a negative margin. Bookcases are in the same situation. Meanwhile, Labels, Paper, and Envelopes sit at 40%+ margins. High sales volume can hide a lot of damage.


## 4. Regional and State Performance

The region-level view is a good starting point, but the real story is at the state level.

In [ ]:
region = df.groupby("region")[["sales","profit"]].sum().reset_index()
region["margin"] = region["profit"] / region["sales"] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Regional Performance")

x = range(len(region))
w = 0.35
axes[0].bar([i-w/2 for i in x], region["sales"]/1000, w, label="Sales ($K)", alpha=0.85)
axes[0].bar([i+w/2 for i in x], region["profit"]/1000, w, label="Profit ($K)", alpha=0.85)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(region["region"])
axes[0].set_ylabel("$ Thousands")
axes[0].set_title("Sales vs Profit by Region")
axes[0].legend()

bars = axes[1].bar(region["region"], region["margin"], width=0.5)
for bar, val in zip(bars, region["margin"]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f"{val:.1f}%", ha="center", fontweight="bold")
axes[1].set_ylabel("Profit Margin (%)")
axes[1].set_title("Margin % by Region")

plt.tight_layout()
plt.show()


In [ ]:
state = df.groupby("state")[["sales","profit"]].sum().reset_index()
state["margin"] = state["profit"] / state["sales"] * 100
state = state.sort_values("profit")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("State-Level Profit")

# Best vs worst 10
combined = pd.concat([state.head(10), state.tail(10)])
axes[0].barh(combined["state"], combined["profit"])
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_title("10 Best vs 10 Worst States (Total Profit)")
axes[0].set_xlabel("Total Profit ($)")

# All states by margin
state_all = state.sort_values("margin")
axes[1].barh(state_all["state"], state_all["margin"])
axes[1].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Profit Margin % — All States")
axes[1].set_xlabel("Margin (%)")
axes[1].tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.show()

print("5 Worst States:")
print(state.head(5)[["state","sales","profit","margin"]].to_string(index=False))


The Central region looks weak because of specific states dragging it down. Texas loses $25.7K despite $170K in sales. Ohio, Pennsylvania, and Illinois have similar problems. These states have decent order volume — the issue is heavy discounting, not low demand.


## 5. Customer Segments

Three types of customers: Consumer, Corporate, and Home Office.

In [ ]:
seg = df.groupby("segment")[["sales","profit"]].sum().reset_index()
seg["margin"] = seg["profit"] / seg["sales"] * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Customer Segment Analysis")

axes[0].pie(seg["sales"], labels=seg["segment"], autopct="%1.1f%%", startangle=140,
            wedgeprops={"edgecolor":"white","linewidth":2})
axes[0].set_title("Share of Sales")

axes[1].pie(seg["profit"], labels=seg["segment"], autopct="%1.1f%%", startangle=140,
            wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("Share of Profit")

bars = axes[2].bar(seg["segment"], seg["margin"], width=0.5)
for bar, val in zip(bars, seg["margin"]):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f"{val:.1f}%", ha="center", fontweight="bold")
axes[2].set_ylabel("Profit Margin (%)")
axes[2].set_title("Margin by Segment")

plt.tight_layout()
plt.show()

print(seg.to_string(index=False))


All three segments are profitable. Consumer is the largest by sales and profit. Home Office has the best margin, likely because smaller buyers receive fewer blanket discounts. No red flags here, but Consumer has the most room to optimise.


## 6. Discount Analysis

This is the most important section. The data shows a clear pattern: discounts above 20% consistently result in a loss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("How Discounts Affect Profit")

axes[0].scatter(df["discount"], df["profit"], alpha=0.3, s=12)
axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")
z = np.polyfit(df["discount"], df["profit"], 1)
x_line = np.linspace(0, df["discount"].max(), 100)
axes[0].plot(x_line, np.poly1d(z)(x_line), color="red", linewidth=2, label="Trend")
axes[0].set_xlabel("Discount Given")
axes[0].set_ylabel("Profit ($)")
axes[0].set_title("Each dot is one order")
axes[0].legend()

df["disc_bucket"] = pd.cut(df["discount"],
    bins=[-0.01, 0, 0.20, 0.50, 1.0],
    labels=["No discount (0%)", "Small (1-20%)", "Heavy (21-50%)", "Massive (51%+)"]
)
bracket = df.groupby("disc_bucket", observed=True)["profit"].mean().reset_index()
bars = axes[1].bar(bracket["disc_bucket"], bracket["profit"])
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Average Profit per Order by Discount Level")
axes[1].set_xlabel("Discount Tier")
axes[1].set_ylabel("Avg Profit ($)")
for bar, val in zip(bars, bracket["profit"]):
    axes[1].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height() + (2 if val >= 0 else -7),
                 f"${val:.0f}", ha="center", fontsize=10, fontweight="bold")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


In [ ]:
heavy = df[df["discount"] > 0.20]

print("Discount Impact Summary")
print(f"Orders with discount above 20%:   {len(heavy):,} ({len(heavy)/len(df)*100:.1f}% of all orders)")
print(f"Total loss from those orders:     ${heavy[heavy['profit']<0]['profit'].sum():,.0f}")
print()
print(f"Avg profit — no discount:         ${df[df['discount']==0]['profit'].mean():.1f}")
print(f"Avg profit — 1 to 20% discount:   ${df[(df['discount']>0)&(df['discount']<=0.2)]['profit'].mean():.1f}")
print(f"Avg profit — 21 to 50% discount:  ${df[(df['discount']>0.2)&(df['discount']<=0.5)]['profit'].mean():.1f}")


The tipping point is 20%. Below it, orders are profitable. Above it, the average order loses $110. This is also the main reason certain states and Furniture as a category underperform — they receive disproportionately high discounts.


In [ ]:
cat_disc = df.groupby("category").agg(
    avg_discount  = ("discount", "mean"),
    profit_margin = ("profit", lambda x: x.sum() / df.loc[x.index,"sales"].sum() * 100)
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(cat_disc["avg_discount"]*100, cat_disc["profit_margin"], s=300, zorder=3)
for _, row in cat_disc.iterrows():
    ax.annotate(row["category"], (row["avg_discount"]*100, row["profit_margin"]),
                textcoords="offset points", xytext=(8, 0), fontsize=11)
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("Average Discount Given (%)")
ax.set_ylabel("Profit Margin (%)")
ax.set_title("Average Discount vs Profit Margin by Category")
plt.tight_layout()
plt.show()


Furniture gets the highest average discounts and has the lowest margin. Technology keeps discounts low and earns the most. The pattern is consistent — discount less, earn more.


## 7. Time Trends and Seasonality

Sales are not uniform throughout the year. Understanding when peaks and dips occur is important for planning.

In [ ]:
monthly = df.groupby(df["order_date"].dt.to_period("M")).agg(
    sales=("sales","sum"), profit=("profit","sum")
).reset_index()
monthly["order_date"] = monthly["order_date"].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle("Monthly Sales and Profit (2014-2017)")

axes[0].plot(monthly["order_date"], monthly["sales"]/1000, linewidth=2)
axes[0].fill_between(monthly["order_date"], monthly["sales"]/1000, alpha=0.2)
axes[0].set_title("Monthly Sales ($K)")
axes[0].set_xticks(monthly["order_date"][::6])
axes[0].tick_params(axis="x", rotation=45, labelsize=8)
axes[0].set_ylabel("Sales ($K)")

axes[1].bar(monthly["order_date"], monthly["profit"]/1000, alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Monthly Profit ($K)")
axes[1].set_xticks(monthly["order_date"][::6])
axes[1].tick_params(axis="x", rotation=45, labelsize=8)
axes[1].set_ylabel("Profit ($K)")

plt.tight_layout()
plt.show()


In [ ]:
quarterly = df.groupby(["year","quarter"])["sales"].sum().reset_index()
pivot = quarterly.pivot(index="year", columns="quarter", values="sales")
pivot.columns = ["Q1","Q2","Q3","Q4"]

annual = df.groupby("year")[["sales","profit"]].sum().reset_index()
annual["sales_growth"] = annual["sales"].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Seasonality and Year-over-Year Growth")

sns.heatmap(pivot/1000, annot=True, fmt=".0f", cmap="YlGn", linewidths=0.5,
            ax=axes[0], cbar_kws={"label":"Sales ($K)"})
axes[0].set_title("Quarterly Sales ($K) — Q4 is the strongest quarter every year")
axes[0].set_xlabel("Quarter")
axes[0].set_ylabel("Year")

x = range(len(annual))
w = 0.35
axes[1].bar([i-w/2 for i in x], annual["sales"]/1000, w, label="Sales ($K)", alpha=0.85)
axes[1].bar([i+w/2 for i in x], annual["profit"]/1000, w, label="Profit ($K)", alpha=0.85)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(annual["year"])
axes[1].set_ylabel("$ Thousands")
axes[1].set_title("Annual Sales and Profit")
axes[1].legend()
for i, row in annual.iterrows():
    if i > 0:
        axes[1].text(i-w/2, row["sales"]/1000 + 5,
                     f"+{row['sales_growth']:.0f}%",
                     ha="center", fontsize=8)

plt.tight_layout()
plt.show()

print("Annual Summary:")
print(annual.to_string(index=False))


Q4 is the strongest quarter every single year — driven by holiday purchasing and corporate year-end budget spending. Q1 is consistently the slowest. The jagged pattern on the monthly chart is not random noise; it is the same seasonal cycle repeating each year.

Sales grew 51% from 2014 to 2017. Profit grew 89% over the same period, meaning the business became more efficient as it scaled.


## 8. Shipping Analysis

Does delivery speed influence how much customers spend or how much profit is made?

In [ ]:
ship = df.groupby("ship_mode").agg(
    avg_days   = ("shipping_days", "mean"),
    avg_sales  = ("sales",  "mean"),
    avg_profit = ("profit", "mean"),
    orders     = ("order_id","count")
).reset_index().sort_values("avg_days")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Shipping Mode Analysis")

axes[0].bar(ship["ship_mode"], ship["avg_days"])
for i, (bar, val) in enumerate(zip(axes[0].patches, ship["avg_days"])):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f"{val:.1f}d", ha="center", fontweight="bold")
axes[0].set_title("Average Delivery Time (days)")
axes[0].set_ylabel("Days")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(ship["ship_mode"], ship["orders"])
axes[1].set_title("Number of Orders per Mode")
axes[1].set_ylabel("Orders")
axes[1].tick_params(axis="x", rotation=15)

axes[2].bar(ship["ship_mode"], ship["avg_profit"])
axes[2].set_title("Average Profit per Order by Mode")
axes[2].set_ylabel("Avg Profit ($)")
axes[2].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print(ship[["ship_mode","avg_days","orders","avg_sales","avg_profit"]].to_string(index=False))


Standard Class handles the most orders and customers are clearly comfortable with the 5-day wait. Shipping mode does not significantly change profit per order — the discount level is a much stronger factor. There is no clear benefit to pushing customers toward faster (and more expensive) shipping options.


## 9. Customer Loyalty

Are customers coming back, and who are the most valuable ones?

In [ ]:
cust = df.groupby("customer_id").agg(
    total_orders = ("order_id", "nunique"),
    total_spent  = ("sales",    "sum"),
    total_profit = ("profit",   "sum"),
    last_order   = ("order_date","max")
).reset_index()

snapshot = df["order_date"].max() + pd.Timedelta(days=1)
cust["days_since"] = (snapshot - cust["last_order"]).dt.days

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Customer Behaviour Overview")

axes[0].hist(cust["total_orders"], bins=15, edgecolor="white")
axes[0].set_title("Orders per Customer")
axes[0].set_xlabel("Number of Orders")
axes[0].set_ylabel("Customers")
axes[0].axvline(cust["total_orders"].median(), color="red", linestyle="--",
                label=f"Median: {cust['total_orders'].median():.0f}")
axes[0].legend()

axes[1].hist(cust["total_spent"], bins=20, edgecolor="white")
axes[1].set_title("Total Spend per Customer ($)")
axes[1].set_xlabel("Total Spent ($)")
axes[1].set_ylabel("Customers")
axes[1].axvline(cust["total_spent"].median(), color="red", linestyle="--",
                label=f"Median: ${cust['total_spent'].median():.0f}")
axes[1].legend()

axes[2].hist(cust["days_since"], bins=20, edgecolor="white")
axes[2].set_title("Days Since Last Purchase")
axes[2].set_xlabel("Days")
axes[2].set_ylabel("Customers")

plt.tight_layout()
plt.show()

print(f"Total customers: {len(cust):,}")
print(f"Avg orders per customer:  {cust['total_orders'].mean():.1f}")
print(f"Avg spend per customer:   ${cust['total_spent'].mean():,.0f}")
print(f"Customers with 5+ orders: {(cust['total_orders']>=5).sum():,} ({(cust['total_orders']>=5).mean()*100:.0f}%)")
print(f"Inactive for 6+ months:   {(cust['days_since']>180).sum():,} ({(cust['days_since']>180).mean()*100:.0f}%)")


In [ ]:
def loyalty_segment(row):
    if row["total_orders"] >= 8 and row["days_since"] <= 90:
        return "Champions"
    elif row["total_orders"] >= 5 and row["days_since"] <= 180:
        return "Loyal"
    elif row["days_since"] <= 60:
        return "New or Recent"
    elif row["total_orders"] >= 5 and row["days_since"] > 180:
        return "At Risk"
    else:
        return "Occasional"

cust["segment"] = cust.apply(loyalty_segment, axis=1)
seg_counts = cust["segment"].value_counts().reset_index()
seg_counts.columns = ["segment","count"]
seg_avg = cust.groupby("segment")["total_spent"].mean().reset_index()
merged = seg_counts.merge(seg_avg, on="segment")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Customer Loyalty Segments")

axes[0].bar(merged["segment"], merged["count"])
axes[0].set_title("Number of Customers per Segment")
axes[0].set_ylabel("Customers")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(merged["segment"], merged["total_spent"])
axes[1].set_title("Average Total Spend by Segment ($)")
axes[1].set_ylabel("Avg Total Spent ($)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

print(merged.to_string(index=False))


Most customers are repeat buyers, which is a good sign. Champions and Loyal customers spend considerably more than the average. The At Risk group — frequent buyers who have gone quiet — represents an opportunity for a targeted re-engagement campaign.


## 10. Summary and Recommendations

Everything from the analysis, brought together in one place.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Superstore — Business Summary", fontsize=14, fontweight="bold")

# Category margin
cat2 = df.groupby("category")[["sales","profit"]].sum().reset_index()
cat2["margin"] = cat2["profit"]/cat2["sales"]*100
bars = axes[0,0].bar(cat2["category"], cat2["margin"], width=0.5)
axes[0,0].set_title("Profit Margin by Category")
axes[0,0].set_ylabel("Margin (%)")
for bar, v in zip(bars, cat2["margin"]):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                   f"{v:.1f}%", ha="center", fontweight="bold")

# Discount bracket
bracket2 = df.groupby("disc_bucket", observed=True)["profit"].mean().reset_index()
axes[0,1].bar(bracket2["disc_bucket"].astype(str), bracket2["profit"])
axes[0,1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[0,1].set_title("Avg Profit by Discount Level")
axes[0,1].set_ylabel("Avg Profit ($)")
axes[0,1].tick_params(axis="x", rotation=15, labelsize=8)

# YoY growth
annual2 = df.groupby("year")[["sales","profit"]].sum().reset_index()
axes[0,2].plot(annual2["year"], annual2["sales"]/1000, "o-", label="Sales", linewidth=2)
axes[0,2].plot(annual2["year"], annual2["profit"]/1000, "o-", label="Profit", linewidth=2)
axes[0,2].set_title("Year-over-Year Growth")
axes[0,2].set_ylabel("$ Thousands")
axes[0,2].legend()

# Worst states
worst5 = df.groupby("state")["profit"].sum().nsmallest(5).reset_index()
axes[1,0].barh(worst5["state"], worst5["profit"])
axes[1,0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[1,0].set_title("Top 5 Loss-Making States")
axes[1,0].set_xlabel("Total Profit ($)")

# Sub-cat best and worst
sub3 = df.groupby("sub-category")[["sales","profit"]].sum().reset_index()
sub3["margin"] = sub3["profit"]/sub3["sales"]*100
top_bottom = pd.concat([sub3.nsmallest(3,"margin"), sub3.nlargest(3,"margin")])
axes[1,1].barh(top_bottom["sub-category"], top_bottom["margin"])
axes[1,1].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[1,1].set_title("Best and Worst Sub-Categories by Margin")
axes[1,1].set_xlabel("Margin (%)")

# Loyalty segments
seg_c2 = cust["segment"].value_counts().reset_index()
seg_c2.columns = ["segment","count"]
axes[1,2].pie(seg_c2["count"], labels=seg_c2["segment"], autopct="%1.0f%%",
              wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[1,2].set_title("Customer Loyalty Mix")

plt.tight_layout()
plt.show()


---

## Recommendations

| Priority | Problem | Finding | Suggested Action |
|---|---|---|---|
| High | Heavy discounts causing losses | Orders above 20% discount average a $110 loss | Cap discounts at 20%; anything above needs approval |
| High | Tables and Bookcases losing money | Negative margins despite high sales volume | Review pricing and reduce bundled discounts on these items |
| Medium | Texas, Ohio, Pennsylvania | Combined losses exceed $60K | Apply stricter discount controls in these states |
| Medium | At Risk customers going quiet | Frequent buyers now inactive | Targeted re-engagement with a modest discount offer |
| Lower | Q4 demand far outpaces Q1 | Q4 is consistently 2.4x larger than Q1 | Pre-stock inventory in Q3, ramp marketing in October |
| Lower | Technology discount discipline | Lowest discounts, highest margins | Apply same pricing discipline to Furniture |


## Key Business Insights

After going through the data across categories, regions, customers, discounts, and time, a few patterns stand out clearly:

* The discount structure is the single biggest issue. Orders with no discount average **$67 profit**. Once the discount crosses **20%**, that flips to an average loss of **$110 per order**. Nearly **19% of all orders are loss-making**, and heavy discounting explains most of them.

* Furniture looks healthy on the surface with high sales volume, but it has only a **2.5% profit margin** — the lowest of the three categories by a wide margin. **Tables alone generated $207K in revenue while losing $17.7K**. The category is effectively being discounted into losses.

* The Central region underperforms not because of low demand, but because states like **Texas, Ohio, and Pennsylvania** are receiving disproportionately high discounts. **Texas recorded $170K in sales while generating a negative profit of $25.7K**.

* On the positive side, the business is genuinely growing. Sales rose **51% from 2014 to 2017**, while profit grew **89%**, showing that efficiency improved alongside revenue. **Q4 consistently drives this growth every year**.

* Most customers are repeat buyers, which provides a strong foundation for long-term growth. The main concern is the segment of previously frequent buyers who have gone inactive — representing potentially recoverable revenue.


## Conclusion

The Superstore is a growing business with a real profitability problem hiding underneath the revenue numbers. The core issue is not the product mix or the customer base — both are solid. The issue is discounting. Too many orders are being approved at discount levels that guarantee a loss, and this is concentrated in specific categories and states.

The good news is that this is fixable. Technology already proves the model works — low discounts, strong margins. Applying that same discipline to Furniture and to high-loss states would meaningfully improve profitability without needing to grow sales at all.
